# Iterative MIN3P speciation by site and soil horizon

For each site and horizon:

1. fix pH from `soil_chem`
2. fix pCO2 from `soil_pco2`
3. charge balance the solution with Cl-
4. adjust total H4SiO4, Al, Ca, and Na until the selected mineral saturation indices are zero
5. omits calcite and leaves total Ca fixed when `soil_chem['caco3_lt_2_mm'] < 0.8`.

The optimized variables are logs of total aqueous component concentrations. All minerals remain inactive (`minequil = .false.`), so MIN3P calculates saturation indices without changing the supplied totals. Because `speciation_o.gen` reports saturation indices to three decimal places, the default target tolerance is 0.002.

In [ ]:
import os
import shutil
import re
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

from byte_util.util import all_sites, single_to_double_float
from min3p.input import InputFile

In [ ]:
# Input data
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'
s3_input_path = f'{s3_base_path}/input-data/processed-data'

soil_pco2 = pd.read_csv(f'{s3_input_path}/soil_pco2.csv', index_col=0)
soil_chem = pd.read_parquet(f'{s3_input_path}/soil_chemical_parameters.parquet')

# MIN3P files and run configuration
speciation_folder = Path('../simulations/met_forcing_rxn/min3p_runs/speciation')
template_path = speciation_folder.parent / 'base/speciation.dat'
min3p_executable = os.environ.get('MIN3P_EXEC')
prefix = 'speciation'

horizons = ['A', 'B', 'C']
si_tolerance = 0.1
tot_base_tolerance = 0.1
max_nfev = 80

# Build list of site/horizon pairs to speciate
site_horizon_pairs = [(site, horizon) for site in all_sites for horizon in horizons]
site_horizon_pairs.remove(('Yolo', 'B'))
site_horizon_pairs.remove(('Pullman', 'C'))

In [ ]:
# Global geochemical assumptions
target_quartz_si = 0.0
target_kfeldspar_si = -1.0

# Sum of totcons of all base cations from Deng et al. (Ca + Mg + K + Na)
default_base_cation_total = 9.74e-4 + 3.50e-4 + 7.42e-5 + 1.83e-3
base_cation_totals = {horizon_pair: default_base_cation_total for horizon_pair in site_horizon_pairs}
base_cation_totals[('HoustonBlack', 'C')] = 0.004  # For HoustonBlack C, assume slightly higher base cations

# Gaines-Thomas log K referenced to Na
#   Ca2+ + 2 Na-X = Ca-X + 2 Na+   logK = 0.7959
#   Mg2+ + 2 Na-X = Mg-X + 2 Na+   logK = 0.6021
#   K+  +   Na-X = K-X  +   Na+    logK = 0.6990
gt_log10_k = {
    'ca+2': 0.7959,
    'mg+2': 0.6021,
    'k+1': 0.6990,
}

exchange_columns = {
    'ca+2': 'ca_exchange_pct_of_cec',
    'mg+2': 'mg_exchange_pct_of_cec',
    'na+1': 'na_exchange_pct_of_cec',
    'k+1': 'k_exchange_pct_of_cec',
}

# Small floor used only to avoid zero exchanger fractions in the
# heterovalent Gaines-Thomas equations
exchange_fraction_floor = 1e-12

# Initial guesses and bounds for h4sio4, co3-2 and al+3
trial_totals = {
    'co3-2': 5.2000e-6,
    'na+1': 1.8300e-3,
    'al+3': 6.7700e-10,
    'h4sio4': 6.8300e-4,
}

component_to_mineral = {
    'h4sio4': 'quartz',
    'al+3': 'k-feld-d-ph',
    'co3-2': 'calcite-ph',
}

bounds = {
    'h4sio4': (1.0e-6, 1.0e-2),
    'al+3': (1.0e-10, 1.0e-3),
    'co3-2': (1.0e-6, 1.0e-2),
    'na+1': (1.0e-6, 1.0e-2),
}

### Define Gaines-Thomas exchanger calculations

In [ ]:
def exchanger_equivalent_fractions(site, horizon):
    """Return normalized Ca/Mg/Na/K Gaines-Thomas equivalent fractions."""
    values = {
        ion: float(soil_chem.loc[(site, horizon), column]) / 100.0
        for ion, column in exchange_columns.items()
    }
    values = {ion: max(value, exchange_fraction_floor) for ion, value in values.items()}
    total = sum(values.values())
    if not np.isfinite(total) or total <= 0:
        raise ValueError(f'Invalid KSSL exchange composition for {site} {horizon}: {values}')
    return {ion: value / total for ion, value in values.items()}


def base_cations_from_na_activity(na_activity, exchanger_fractions):
    """Return Ca/Mg/Na/K activities implied by the MIN3P Na-X reactions.

    For divalent M = Ca or Mg:
        M2+ + 2 Na-X = M-X + 2 Na+
        K_M/Na = E_M * a_Na^2 / (E_Na^2 * a_M)

    Therefore:
        a_M = E_M * a_Na^2 / (K_M/Na * E_Na^2)

    For K:
        K+ + Na-X = K-X + Na+
        K_K/Na = E_K * a_Na / (E_Na * a_K)

    Therefore:
        a_K = E_K * a_Na / (K_K/Na * E_Na)
    """
    e = exchanger_fractions
    ena = e['na+1']
    na = float(na_activity)

    ca = e['ca+2'] * na**2 / (10.0**gt_log10_k['ca+2'] * ena**2)
    mg = e['mg+2'] * na**2 / (10.0**gt_log10_k['mg+2'] * ena**2)
    k = e['k+1'] * na / (10.0**gt_log10_k['k+1'] * ena)

    return {'ca+2': ca, 'mg+2': mg, 'na+1': na, 'k+1': k}

## Read total concentrations and mineral saturation indices

In [ ]:
def _read_gen_section(text, start, end):
    pattern = rf"{re.escape(start)}\s*\n[-\s]*\n(?P<body>.*?)(?=\n{re.escape(end)})"
    match = re.search(pattern, text, flags=re.S | re.I)
    if match is None:
        raise ValueError(f"Could not find section {start!r}")
    return match.group('body')


def read_speciation_gen(filename):
    """Read total aqueous component concentrations and mineral saturation indices.

    Parameters
    ----------
    filename : str or Path
        MIN3P generic output file, normally ``speciation_o.gen``.

    Returns
    -------
    totals : pandas.Series
        Total aqueous component concentrations in mol/L H2O.
    saturation_indices : pandas.Series
        Mineral saturation indices.
    """
    filename = Path(filename)
    text = filename.read_text(errors='replace')

    total_body = _read_gen_section(text, 'total concentrations - aqueous phase:', 'components as species in solution:')
    totals = {}
    for line in total_body.splitlines():
        match = re.match(r"^\s*(\S+)\s+([+-]?(?:\d+(?:\.\d*)?|\.\d+)[EeDd][+-]?\d+)\s*$", line)
        if match is not None:
            totals[match.group(1).lower()] = float(match.group(2).replace('D', 'E').replace('d', 'e'))

    mineral_pattern = r"(?ms)^minerals:\s*\n-+\s*\nmineral\s+SI\s+phi\s+area\s*\n-+\s*\n(?P<body>.*?)(?=^charge balance:)"
    mineral_matches = list(re.finditer(mineral_pattern, text))
    if not mineral_matches:
        raise ValueError(f"Could not find the final mineral saturation-index table in {filename}")

    saturation_indices = {}
    for line in mineral_matches[-1].group('body').splitlines():
        match = re.match(r"^\s*(\S+)\s+([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s+", line)
        if match is not None:
            saturation_indices[match.group(1).lower()] = float(match.group(2))

    alkalinity_results = _read_gen_section(text, 'results:', 'carbonate alkalinity:')
    alkalinity, ionic_strength = None, None
    for line in alkalinity_results.splitlines():
        if 'ionic strength' in line:
            ionic_strength = float(line.split()[-1].strip())
        elif 'alkalinity:' in line:
            alkalinity = float(line.split('=')[1].replace('eq/L', '').strip())
    master_variables = {'alkalinity': alkalinity, 'ionic_strength': ionic_strength}

    if not totals:
        raise ValueError(f"No total concentrations were parsed from {filename}")
    if not saturation_indices:
        raise ValueError(f"No saturation indices were parsed from {filename}")

    totals = pd.Series(totals, name='total_concentration_mol_l', dtype=float)
    saturation_indices = pd.Series(saturation_indices, name='saturation_index', dtype=float)
    return totals, saturation_indices, master_variables

## Input-file and MIN3P execution helpers

In [ ]:
def set_concentration_record(zone, components, component, value, input_type):
    """Replace one concentration-input record while preserving its trailing comment."""
    record_index = components.index(component)
    if input_type == 'ph':
        value_string = f'{float(value):.2f}'
    else:
        value_string = single_to_double_float(f'{float(value):.4e}')
    zone.concentration_input.records[record_index].replace_content(f"{value_string:<16} '{input_type}'")


def write_speciation_input(template_path, output_path, ph, pco2, totals, equil_calcite=False):
    """Write one MIN3P batch input for a trial set of total concentrations."""

    infile = InputFile.load(template_path.name, path=template_path.parent)
    zone = infile.initial_conditions_local_geochemistry.zones[0]
    components = infile.geochemical_system.components

    set_concentration_record(zone, components, 'h+1', ph, 'ph')
    for component, value in totals.items():
        if not equil_calcite and component == 'co3-2':
            # If we aren't equilibrating with calcite, then fix the pco2 value
            set_concentration_record(zone, components, 'co3-2', pco2, 'pco2')
        else:
            set_concentration_record(zone, components, component, value, 'free')

    infile.save(output_path)


def run_min3p(run_dir, executable_path=min3p_executable, prefix=prefix):
    """Run MIN3P and return the path to its generic output file."""
    run_dir = Path(run_dir)
    output_path = run_dir / f'{prefix}_o.gen'
    output_path.unlink(missing_ok=True)

    process = subprocess.run([executable_path], cwd=run_dir, text=True, capture_output=True)
    (run_dir / 'min3p.stdout').write_text(process.stdout)
    (run_dir / 'min3p.stderr').write_text(process.stderr)

    if process.returncode != 0:
        raise RuntimeError(f'MIN3P returned code {process.returncode}; see {run_dir / "min3p.stderr"}')
    if not output_path.exists():
        raise FileNotFoundError(f'MIN3P did not create {output_path}')
    if 'normal exit' not in output_path.read_text(errors='replace').lower():
        raise RuntimeError(f'MIN3P did not exit normally; inspect {output_path} and {run_dir / "min3p.stderr"}')
    return output_path

In [ ]:
def speciate_site_horizon(site, horizon, equil_calcite, save_speciation_gen=False):
    """Construct one deterministic initial solution and evaluate it with MIN3P.

    At each evaluation, Ca/Mg/K are calculated from Na using the KSSL exchanger fractions and
    Gaines-Thomas coefficients. MIN3P then speciates the solution, and the optimizer requires
    the sum of total base cation concentrations (totcons, not species concentrations) to equal
    `base_cation_total_mol_l`"""
    ph = float(soil_chem.loc[(site, horizon), 'pH'])
    pco2 = 10.0 ** soil_pco2.loc[site, 'soilco2_log10atm']
    base_cation_total_mol_l = base_cation_totals[(site, horizon)]
    if horizon == 'A':
        pco2 /= 2.0
    exchanger_fractions = exchanger_equivalent_fractions(site, horizon)

    adjustable = ['na+1', 'h4sio4', 'al+3'] + (['co3-2'] if equil_calcite else [])
    target_minerals = ['quartz', 'k-feld-d-ph'] + (['calcite-ph'] if equil_calcite else [])
    target_si = np.array([target_quartz_si, target_kfeldspar_si] + ([0.0] if equil_calcite else []))

    x0 = np.log10([trial_totals[c] for c in adjustable])
    lower = np.log10([bounds[c][0] for c in adjustable])
    upper = np.log10([bounds[c][1] for c in adjustable])

    run_dir = speciation_folder / site / horizon
    run_dir.mkdir(parents=True, exist_ok=True)
    input_path = run_dir / f'{prefix}.dat'
    history = []

    def build_trial(log_totals):
        trial = dict(zip(adjustable, 10.0**np.asarray(log_totals)))
        base_cations = base_cations_from_na_activity(trial.pop('na+1'), exchanger_fractions)
        totals = {**base_cations, **trial}
        if not equil_calcite:
            totals['co3-2'] = trial_totals['co3-2']
        return totals

    def residual(log_totals):
        totals = build_trial(log_totals)

        write_speciation_input(template_path, input_path, ph, pco2, totals, equil_calcite=equil_calcite)
        output_path = run_min3p(run_dir)
        output_totals, si, master_variables = read_speciation_gen(output_path)

        # Save a copy of speciation_o.gen
        if save_speciation_gen:
            shutil.copy(output_path, output_path.parent / f'speciation_{len(history):02d}.gen')

        si_residuals = si.reindex(target_minerals).to_numpy(float) - target_si
        total_base = output_totals.reindex(['ca+2', 'mg+2', 'na+1', 'k+1']).sum()
        base_residual = np.log10(total_base / base_cation_total_mol_l)
        r = np.r_[si_residuals, base_residual]

        history.append({
            **{f'log10_input_{c}': x for c, x in zip(adjustable, log_totals)},
            **{f'input_{c}_mol_l': totals[c] for c in ['ca+2', 'mg+2', 'na+1', 'k+1']},
            **{f'si_{m}': si[m] for m in target_minerals},
            'total_base_cations_mol_l': total_base,
            'base_cation_log_resdual': base_residual,
            **master_variables,
        })
        return r

    try:
        result = least_squares(residual, x0, bounds=(lower, upper), diff_step=2e-2,
                               xtol=1e-4, ftol=1e-4, gtol=1e-4, max_nfev=max_nfev,)
    except ValueError as e:
        # If ValueError, save history before exiting
        pd.DataFrame(history).to_csv(run_dir / 'iteration_history.csv', index=False)
        raise e

    final_input = build_trial(result.x)
    write_speciation_input(template_path, input_path, ph, pco2, final_input, equil_calcite=equil_calcite)
    output_path = run_min3p(run_dir)
    output_totals, si, master_variables = read_speciation_gen(output_path)

    target_residual = si.reindex(target_minerals).to_numpy(float) - target_si
    max_abs_si_error = float(np.max(np.abs(target_residual)))
    total_base = output_totals.reindex(['ca+2', 'mg+2', 'na+1', 'k+1']).sum()
    base_log_error = np.log10(total_base / base_cation_total_mol_l)
    pd.DataFrame(history).to_csv(run_dir / 'iteration_history.csv', index=False)

    summary = {
        'site': site,
        'horizon': horizon,
        'pH': ph,
        'pCO2_atm': pco2,
        'caco3_lt_2_mm': soil_chem.loc[(site, horizon), 'caco3_lt_2_mm'],
        'equil_calcite': equil_calcite,
        'optimizer_success': bool(result.success),
        'converged': bool(result.success
                          and max_abs_si_error <= si_tolerance
                          and abs(base_log_error) <= tot_base_tolerance),
        'nfev': int(result.nfev),
        'max_abs_target_si_error': max_abs_si_error,
        'total_base_cations_mol_l': total_base,
        'base_cation_log_error': base_log_error,
        'optimizer_message': result.message,
        'alkalinity_eq_l': master_variables['alkalinity'],
        'ionic_strength': master_variables['ionic_strength'],
    }
    summary.update({f'kssl_exchange_fraction_{ion}': x for ion, x in exchanger_fractions.items()})
    summary.update({f'input_{ion}_mol_l': x for ion, x in final_input.items()})
    summary.update({f'total_{ion}_mol_l': x for ion, x in output_totals.items()})
    summary.update({f'si_{mineral}': x for mineral, x in si.items()})
    return summary


## Run every available site/horizon combination

In [ ]:
rerun_speciation = True
results_path = Path(speciation_folder)
equil_calcite_by_site = {'Cecil_A': False,
                         'Cecil_B': False,
                         'Cecil_C': False,
                         'Flanagan_A': False,
                         'Flanagan_B': False,
                         'Flanagan_C': True,
                         'HoustonBlack_A': True,
                         'HoustonBlack_B': True,
                         'HoustonBlack_C': True,
                         'Kalamazoo_A': False,
                         'Kalamazoo_B': False,
                         'Kalamazoo_C': True,
                         'Kuma_A': False,
                         'Kuma_B': True,
                         'Kuma_C': True,
                         'Palouse_A': False,
                         'Palouse_B': False,
                         'Palouse_C': False,
                         'Pullman_A': False,
                         'Pullman_B': True,
                         'Yolo_A': False,
                         'Yolo_C': True,}

if rerun_speciation:
    print(f'Running {len(site_horizon_pairs)} site/horizon speciation calculations')

    summaries = []
    for site, horizon in site_horizon_pairs:
        print(f'{site}: {horizon} horizon')
        ph = soil_chem.loc[(site, horizon), 'pH']
        try:
            summaries.append(speciate_site_horizon(site, horizon,
                                                   equil_calcite=equil_calcite_by_site[f'{site}_{horizon}']))
        except ValueError:
            print(f'Could not speciate {site}: {horizon} horizon')

    speciation_results = pd.DataFrame(summaries).set_index(['site', 'horizon']).sort_index()

    results_path.mkdir(parents=True, exist_ok=True)
    speciation_results.to_parquet(results_path / 'speciation_results.parquet')

else:
    speciation_results = pd.read_parquet(results_path / 'speciation_results.parquet')

print(speciation_results[['pH', 'pCO2_atm', 'caco3_lt_2_mm', 'equil_calcite',
                          'converged', 'nfev', 'max_abs_target_si_error']])

## Merge the equilibrium total concentrations

In [ ]:
# Add the equilibrium total concentrations to a copy of soil_chem
initial_chem = pd.DataFrame(index=soil_chem.index, columns=['co3-2_mol.L'], dtype=float)
for component in ['co3-2', 'mg+2', 'ca+2', 'na+1', 'k+1', 'al+3', 'h4sio4', 'cl-1']:
    column = f'total_{component}_mol_l'
    if column in speciation_results.columns:
        initial_chem.loc[speciation_results.index, f'{component}_mol.L'] = speciation_results[column]
copy_cols = ['pCO2_atm', 'equil_calcite', 'alkalinity_eq_l', 'ionic_strength']
for col in copy_cols:
    initial_chem.loc[speciation_results.index, col] = speciation_results[col]

# Double check total base ion concentrations
initial_chem['tot_base'] = 0.
base_cat_cols = ['ca+2_mol.L', 'mg+2_mol.L', 'na+1_mol.L', 'k+1_mol.L']
for idx in initial_chem.index:
    initial_chem.loc[idx, 'tot_base'] = initial_chem.loc[idx, base_cat_cols].sum()

if rerun_speciation:
    initial_chem.to_parquet(results_path / 'initial_aqueous_chem.parquet')

# Compare the change from original speciation results to the new results
print('What changed from v1 speciation to this speciation:')
v1_chem = pd.read_parquet(results_path / 'initial_aqueous_chem_v1.parquet')
for col in [c for c in initial_chem.columns if c.endswith('_mol.L')]:
    print(f'{col.split('_')[0]}')
    for site, horizon in site_horizon_pairs:
        new_value = initial_chem.loc[(site, horizon), col]
        old_value = v1_chem.loc[(site, horizon), col]
        pct_diff = np.abs(100.0 * (new_value - old_value)/old_value)
        if pct_diff > 500:
            print(f'    {site}, {horizon}: {old_value} --> {new_value}')

## Plot concentrations across sites/hoirzons

In [ ]:
import matplotlib.pyplot as plt

offsets = {'A': -0.28, 'B': 0.00, 'C': 0.28}
horizon_colors = {'A': '#7A5C47', 'B': '#B97A57', 'C': '#CDBFAE'}
width = 0.22

def add_horizon_bars(ax, parameter):
    """Add A, B, and C boxplots for each soil series."""
    for horizon_name in horizons:
        values = initial_chem.loc[(all_sites, horizon_name), parameter]

        xpos = [i + offsets[horizon_name] for i in range(len(all_sites))]

        ax.bar(xpos, values, width=width, color=horizon_colors[horizon_name])

    ax.set(xlim=(-0.6, len(all_sites) - 0.4), xticks=range(len(all_sites)), xticklabels=all_sites)
    ax.tick_params(axis='x', rotation=30)

cols_to_plot = ['mg+2_mol.L', 'ca+2_mol.L', 'k+1_mol.L', 'na+1_mol.L', 'al+3_mol.L', 'h4sio4_mol.L',
                'tot_base', 'alkalinity_eq_l', 'ionic_strength']

fig, ax = plt.subplots(len(cols_to_plot), sharex=True, figsize=(8, 12), tight_layout=True)

for i, col in enumerate(cols_to_plot):
    add_horizon_bars(ax[i], col)
    ax[i].set(ylabel=col)
ax[cols_to_plot.index('tot_base')].axhline(default_base_cation_total, color='k', ls='--')